# Test de Arquitectura Refactorizada

Este notebook prueba la nueva arquitectura modular del Spatial Iteration Engine después de la refactorización.

## Cambios principales:
- Interfaces unificadas en `ports/processors.py`
- Pipelines reorganizados en `application/pipeline/`
- Orquestador de pipeline (`PipelineOrchestrator`)
- Servicios extraídos (ErrorHandler, RetryManager, FrameBuffer)
- Adaptadores reorganizados (`adapters/processors/`)

## Nuevos imports:
- `from ascii_stream_engine.adapters.processors import ...` (en lugar de `adapters.filters` y `adapters.analyzers`)
- `from ascii_stream_engine.application.pipeline import ...` (pipelines unificados)


In [1]:
# Configurar el path para importar el módulo
import sys
from pathlib import Path

# Agregar el directorio raíz del proyecto al PYTHONPATH
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Python: {sys.version}")
print(f"Project root: {project_root}")
print(f"PYTHONPATH incluye: {project_root in [Path(p) for p in sys.path]}")
print("\n=== Verificando nuevos imports ===")

# Imports principales
try:
    from ascii_stream_engine import (
        EngineConfig,
        StreamEngine,
        AnalyzerPipeline,
        FilterPipeline,
    )
    print("✓ Imports principales OK")
except ImportError as e:
    print(f"✗ Error en imports principales: {e}")

# Imports de processors (nueva ubicación)
try:
    from ascii_stream_engine.adapters.processors import (
        BaseFilter,
        BaseAnalyzer,
        BrightnessFilter,
        InvertFilter,
        EdgeFilter,
        DetailBoostFilter,
        FaceHaarAnalyzer,
    )
    print("✓ Imports de processors OK")
except ImportError as e:
    print(f"✗ Error en imports de processors: {e}")

# Imports de pipelines
try:
    from ascii_stream_engine.application.pipeline import (
        AnalyzerPipeline,
        FilterPipeline,
        TransformationPipeline,
        TrackingPipeline,
    )
    print("✓ Imports de pipelines OK")
except ImportError as e:
    print(f"✗ Error en imports de pipelines: {e}")

# Imports de orquestación
try:
    from ascii_stream_engine.application.orchestration import (
        PipelineOrchestrator,
        StageExecutor,
    )
    print("✓ Imports de orquestación OK")
except ImportError as e:
    print(f"✗ Error en imports de orquestación: {e}")

# Imports de servicios
try:
    from ascii_stream_engine.application.services import (
        ErrorHandler,
        RetryManager,
        FrameBuffer,
    )
    print("✓ Imports de servicios OK")
except ImportError as e:
    print(f"✗ Error en imports de servicios: {e}")

# Imports de adaptadores
try:
    from ascii_stream_engine.adapters.sources import OpenCVCameraSource
    from ascii_stream_engine.adapters.renderers import AsciiRenderer
    from ascii_stream_engine.adapters.outputs import FfmpegUdpOutput
    print("✓ Imports de adaptadores OK")
except ImportError as e:
    print(f"✗ Error en imports de adaptadores: {e}")

print("\n=== Todos los imports verificados ===")


Python: 3.13.11 (main, Dec  8 2025, 11:43:54) [GCC 15.2.0]
Project root: /home/fissure/repos/Spatial-Iteration-Engine
PYTHONPATH incluye: True

=== Verificando nuevos imports ===
✓ Imports principales OK
✓ Imports de processors OK
✓ Imports de pipelines OK
✓ Imports de orquestación OK
✓ Imports de servicios OK
✓ Imports de adaptadores OK

=== Todos los imports verificados ===


## Test 1: Crear pipelines con la nueva estructura


In [ ]:
# Crear pipelines usando la nueva estructura
from ascii_stream_engine.adapters.processors import (
    BrightnessFilter,
    InvertFilter,
    EdgeFilter,
)

# Crear FilterPipeline
filters = FilterPipeline([
    BrightnessFilter(),
    InvertFilter(),
    EdgeFilter(),
])

print(f"✓ FilterPipeline creado con {len(filters.filters)} filtros")
print(f"  Filtros: {[f.name for f in filters.filters]}")

# Crear AnalyzerPipeline
analyzers = AnalyzerPipeline()
print(f"✓ AnalyzerPipeline creado")

# Verificar que implementan ProcessorPipeline
from ascii_stream_engine.ports.processors import ProcessorPipeline

# Verificar que tienen los métodos requeridos del protocolo
has_add = hasattr(filters, 'add')
has_remove = hasattr(filters, 'remove')
has_process = hasattr(filters, 'process')
has_has_any = hasattr(filters, 'has_any')

print(f"✓ FilterPipeline tiene métodos del protocolo: add={has_add}, remove={has_remove}, process={has_process}, has_any={has_has_any}")
#print(f"✓ FilterPipeline implementa ProcessorPipeline: {isinstance(filters, ProcessorPipeline)}")
#print(f"✓ AnalyzerPipeline implementa ProcessorPipeline: {isinstance(analyzers, ProcessorPipeline)}")


✓ FilterPipeline creado con 3 filtros
  Filtros: ['brightness', 'invert', 'edges']
✓ AnalyzerPipeline creado
✓ FilterPipeline tiene métodos del protocolo: add=True, remove=True, process=True, has_any=True


## Test 2: Crear configuración y verificar estructura


In [3]:
# Crear configuración
config = EngineConfig(
    fps=20,
    grid_w=80,
    grid_h=40,
    host="127.0.0.1",
    port=1234,
    frame_buffer_size=2,
    enable_events=True,
)

print(f"✓ Configuración creada")
print(f"  FPS: {config.fps}")
print(f"  Grid: {config.grid_w}x{config.grid_h}")
print(f"  Output: {config.host}:{config.port}")
print(f"  Events: {config.enable_events}")


✓ Configuración creada
  FPS: 20
  Grid: 80x40
  Output: 127.0.0.1:1234
  Events: True


## Test 3: Crear componentes del engine (sin iniciar)


In [4]:
# Crear componentes (usando dummies si no hay cámara disponible)
import numpy as np

# Dummy source para testing
class DummySource:
    def open(self):
        pass
    def read(self):
        return np.zeros((480, 640, 3), dtype=np.uint8)
    def close(self):
        pass

# Dummy renderer
class DummyRenderer:
    def output_size(self, config):
        return (config.grid_w, config.grid_h)
    def render(self, frame, config, analysis=None):
        from ascii_stream_engine.domain.types import RenderFrame
        return RenderFrame(data=b"dummy", width=config.grid_w, height=config.grid_h)

# Dummy sink
class DummySink:
    def open(self, config, output_size):
        pass
    def write(self, frame):
        pass
    def close(self):
        pass

# Crear engine con la nueva arquitectura
source = DummySource()
renderer = DummyRenderer()
sink = DummySink()

engine = StreamEngine(
    source=source,
    renderer=renderer,
    sink=sink,
    config=config,
    analyzers=analyzers,
    filters=filters,
    enable_profiling=True,
)

print("✓ StreamEngine creado con nueva arquitectura")
print(f"  Analyzers: {len(engine.analyzers)}")
print(f"  Filters: {len(engine.filters)}")
print(f"  Profiling: {engine.profiler.enabled}")
print(f"  Event Bus: {engine.get_event_bus() is not None}")

# Verificar servicios internos
print("\n=== Verificando servicios internos ===")
print(f"  Profiler disponible: {engine.profiler is not None}")
print(f"  Metrics disponible: {engine.metrics is not None}")


✓ StreamEngine creado con nueva arquitectura
  Analyzers: 0
  Filters: 3
  Profiling: True
  Event Bus: True

=== Verificando servicios internos ===
  Profiler disponible: True
  Metrics disponible: True


In [5]:
# Verificar que los imports antiguos aún funcionan (con warnings)
import warnings

print("=== Test de compatibilidad hacia atrás ===")

# Capturar warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    
    # Import antiguo de filters
    from ascii_stream_engine.adapters.filters import BrightnessFilter as OldBrightnessFilter
    
    if w:
        print(f"✓ Import antiguo funciona (con warning: {w[0].message})")
    else:
        print("✓ Import antiguo funciona (sin warning)")
    
    # Verificar que es la misma clase
    from ascii_stream_engine.adapters.processors import BrightnessFilter as NewBrightnessFilter
    print(f"✓ Misma clase: {OldBrightnessFilter is NewBrightnessFilter}")

print("\n=== Compatibilidad verificada ===")


=== Test de compatibilidad hacia atrás ===
✓ Import antiguo funciona (con warning: Importar desde adapters.filters está deprecado. Usa 'from ascii_stream_engine.adapters.processors import ...' en su lugar.)
✓ Misma clase: True

=== Compatibilidad verificada ===


## Test 5: Crear engine completo con cámara real


In [12]:
# Crear engine completo con cámara real usando la nueva arquitectura
from ascii_stream_engine.adapters.sources import OpenCVCameraSource
from ascii_stream_engine.adapters.renderers import AsciiRenderer
from ascii_stream_engine.adapters.outputs import FfmpegUdpOutput

# Configuración optimizada para testing
config = EngineConfig(
    fps=20,
    grid_w=80,
    grid_h=40,
    host="239.0.0.1",
    port=1234,
    frame_buffer_size=2,
    enable_events=True,
  #  enable_profiling=True,
)

# Crear pipelines con filtros
filters = FilterPipeline([
    BrightnessFilter(),
    InvertFilter(),
])

# Crear source, renderer y sink
source = OpenCVCameraSource(0)  # Cámara 0
renderer = AsciiRenderer()
sink = FfmpegUdpOutput()

# Crear engine con nueva arquitectura
engine = StreamEngine(
    source=source,
    renderer=renderer,
    sink=sink,
    config=config,
    analyzers=analyzers,
    filters=filters,
    enable_profiling=True,
)

print("✓ Engine creado con nueva arquitectura")
print(f"  Source: {type(source).__name__}")
print(f"  Renderer: {type(renderer).__name__}")
print(f"  Sink: {type(sink).__name__}")
print(f"  Filters: {len(engine.filters)}")
print(f"  Analyzers: {len(engine.analyzers)}")
print(f"  Profiling: {engine.profiler.enabled}")
print(f"  Event Bus: {engine.get_event_bus() is not None}")

print("\n⚠️  Para iniciar el engine, ejecuta: engine.start(blocking=True)")
print("⚠️  Para detener: engine.stop() o Ctrl+C")


✓ Engine creado con nueva arquitectura
  Source: OpenCVCameraSource
  Renderer: AsciiRenderer
  Sink: FfmpegUdpOutput
  Filters: 2
  Analyzers: 0
  Profiling: True
  Event Bus: True

⚠️  Para iniciar el engine, ejecuta: engine.start(blocking=True)
⚠️  Para detener: engine.stop() o Ctrl+C


## Test 6: Iniciar engine y verificar funcionamiento


In [13]:
# Iniciar el engine (ejecutar solo si quieres probar en tiempo real)
# Descomenta las siguientes líneas para iniciar:

# import time
# print("Iniciando engine...")
# engine.start(blocking=False)  # No bloqueante para poder detenerlo
# 
# try:
#     # Esperar unos segundos para ver el stream
#     time.sleep(10)
#     print("\n✓ Engine funcionando correctamente")
#     print(f"  Frames procesados: {engine.metrics.total_frames}")
#     print(f"  Errores: {engine.metrics.total_errors}")
#     
#     # Ver estadísticas de profiling
#     if engine.profiler.enabled:
#         print("\n=== Estadísticas de Profiling ===")
#         stats = engine.get_profiling_stats()
#         for phase, data in stats.items():
#             print(f"  {phase}: avg={data.get('avg', 0)*1000:.2f}ms")
# finally:
#     engine.stop()
#     print("\n✓ Engine detenido")

print("⚠️  Descomenta el código arriba para probar el engine en tiempo real")
print("   o ejecuta: engine.start(blocking=True)")


⚠️  Descomenta el código arriba para probar el engine en tiempo real
   o ejecuta: engine.start(blocking=True)


In [14]:
engine.start(blocking=False)

In [15]:
engine.stop()

In [8]:
stats = engine.get_profiling_stats()

In [9]:
stats

{'capture': {'count': 688,
  'total_time': 0.001650371999971867,
  'avg_time': 2.398796511587016e-06,
  'min_time': 1.0889999941809947e-06,
  'max_time': 1.5622999995912323e-05,
  'std_dev': 1.1822607193798312e-06},
 'analysis': {'count': 688,
  'total_time': 0.004854816999852574,
  'avg_time': 7.056420057925252e-06,
  'min_time': 4.1400000156954775e-06,
  'max_time': 5.2299000003586116e-05,
  'std_dev': 4.239968261658366e-06},
 'transformation': {'count': 688,
  'total_time': 0.0007413220006355914,
  'avg_time': 1.077502907900569e-06,
  'min_time': 6.029999894963112e-07,
  'max_time': 5.990999994764934e-06,
  'std_dev': 4.801739130922023e-07},
 'filtering': {'count': 688,
  'total_time': 0.415859475999639,
  'avg_time': 0.000604446912790173,
  'min_time': 0.00035507300000858777,
  'max_time': 0.002217176000016252,
  'std_dev': 0.0002698330478327098},
 'rendering': {'count': 688,
  'total_time': 32.380723109999735,
  'avg_time': 0.04706500452034845,
  'min_time': 0.02636474999999905,
 

## Test 7: Verificar servicios y orquestación


In [10]:
# Verificar servicios extraídos
from ascii_stream_engine.application.services import (
    ErrorHandler,
    RetryManager,
    FrameBuffer,
)

print("=== Verificando servicios ===")

# ErrorHandler
error_handler = ErrorHandler()
print(f"✓ ErrorHandler creado")
print(f"  Error count: {error_handler.get_error_count()}")

# RetryManager
retry_manager = RetryManager()
print(f"✓ RetryManager creado")
print(f"  Max camera retries: {retry_manager.max_camera_retries}")
print(f"  Max UDP retries: {retry_manager.max_udp_retries}")

# FrameBuffer
frame_buffer = FrameBuffer(max_size=5)
print(f"✓ FrameBuffer creado")
print(f"  Max size: {frame_buffer.max_size}")

# Test agregar frame
import numpy as np
test_frame = np.zeros((100, 100, 3), dtype=np.uint8)
frame_buffer.add(test_frame)
print(f"  After add - Size: {frame_buffer.size()}, Empty: {frame_buffer.is_empty()}")

latest = frame_buffer.get_latest()
if latest:
    frame, timestamp = latest
    print(f"  Latest frame shape: {frame.shape}, timestamp: {timestamp}")

# Verificar orquestación
from ascii_stream_engine.application.orchestration import PipelineOrchestrator, StageExecutor

print("\n=== Verificando orquestación ===")
print(f"✓ PipelineOrchestrator disponible")
print(f"✓ StageExecutor disponible")

# El orquestador se crea internamente cuando se inicia el engine
# pero podemos verificar que la clase existe
executor = StageExecutor(timeout=None, retry_count=0)
print(f"✓ StageExecutor creado")
print(f"  Timeout: {executor.timeout}")
print(f"  Retry count: {executor.retry_count}")

print("\n=== Todos los servicios funcionan correctamente ===")


=== Verificando servicios ===
✓ ErrorHandler creado
  Error count: 0
✓ RetryManager creado
  Max camera retries: 5
  Max UDP retries: 3
✓ FrameBuffer creado
  Max size: 5
  After add - Size: 1, Empty: False
  Latest frame shape: (100, 100, 3), timestamp: 1770397191.1132636

=== Verificando orquestación ===
✓ PipelineOrchestrator disponible
✓ StageExecutor disponible
✓ StageExecutor creado
  Timeout: None
  Retry count: 0

=== Todos los servicios funcionan correctamente ===


## Resumen de la refactorización

### ✅ Cambios implementados:

1. **Interfaces unificadas** (`ports/processors.py`)
   - `FrameProcessor`, `Filter`, `Analyzer`, `ProcessorPipeline`

2. **Pipelines reorganizados** (`application/pipeline/`)
   - Todos los pipelines en un solo lugar
   - Estructura unificada

3. **Orquestación** (`application/orchestration/`)
   - `PipelineOrchestrator`: Orquesta el flujo completo
   - `StageExecutor`: Ejecuta etapas con manejo de errores

4. **Servicios** (`application/services/`)
   - `ErrorHandler`: Manejo centralizado de errores
   - `RetryManager`: Reintentos y reconexiones
   - `FrameBuffer`: Buffer thread-safe de frames

5. **Adaptadores reorganizados** (`adapters/processors/`)
   - `filters/` y `analyzers/` ahora en `processors/`
   - Compatibilidad hacia atrás mantenida

6. **Engine refactorizado**
   - Reducido de ~697 a ~400 líneas
   - Delegación al orquestador
   - Uso de servicios

### 📊 Métricas:
- **Líneas de código en engine**: Reducidas significativamente
- **Separación de responsabilidades**: ✅ Mejorada
- **Extensibilidad**: ✅ Mejorada
- **Mantenibilidad**: ✅ Mejorada
- **Compatibilidad**: ✅ Mantenida

### 🎯 Próximos pasos:
- Probar con cámara real (ya configurado arriba)
- Agregar nuevos procesadores
- Extender con nuevos outputs
- Agregar tracking y transformaciones


## Test 4: Verificar compatibilidad hacia atrás
